In [ ]:
import pandas as pd

In [ ]:
datos2023 = pd.read_csv('datos2023.csv')
datos2023['DIRECCION'] = datos2023['DIRECCION'].astype('string')

In [ ]:
datos2023['CÓDIGO DIVIPOLA'] = (
    datos2023['CÓDIGO DEPARTAMENTO'].astype(str).str.zfill(2) +
    datos2023['CÓDIGO MUNICIPIO'].astype(str).str.zfill(3)
).astype('int64')

In [ ]:
poblacion2023 = pd.read_excel('Poblacion2023TotalMunicipios.xlsx')
poblacion2023['CÓDIGO DIVIPOLA'] = pd.to_numeric(
    poblacion2023['CÓDIGO DIVIPOLA'], errors='coerce').astype('Int64')
datos2023['CÓDIGO DIVIPOLA'] = pd.to_numeric(
    datos2023['CÓDIGO DIVIPOLA'], errors='coerce').astype('Int64')

In [ ]:
dfmerge = poblacion2023[['CÓDIGO DIVIPOLA', 'Departamento',
                          'MUNICIPIO', 'PoblacionTotal2023']].copy()

df2023total = datos2023.merge(
    dfmerge,
    on='CÓDIGO DIVIPOLA',
    how='left',
    suffixes=('', '_pob')
)

df2023total['DEPARTAMENTO'] = df2023total['Departamento'].fillna(
    df2023total['DEPARTAMENTO'])
df2023total['MUNICIPIO'] = df2023total['MUNICIPIO_pob'].fillna(
    df2023total['MUNICIPIO'])

df2023total = df2023total.rename(
    columns={'PoblacionTotal2023': 'PoblacionTotalMunicipio2023'})

df2023total = df2023total.drop(columns=['Departamento', 'MUNICIPIO_pob'])

In [ ]:
municipio2023 = df2023total.groupby(
    ['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO'],
    as_index=False
).agg({
    'CANTIDAD': 'sum',
    'PoblacionTotalMunicipio2023': 'max'
})

municipio2023.columns = ['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO',
                          'comparendos', 'poblacion']

municipio2023['poblacion'] = municipio2023['poblacion'].replace(0, pd.NA)

municipio2023['tasacomparendos100k'] = (
    municipio2023['comparendos'] / municipio2023['poblacion'] * 100000)

In [ ]:
df2023total = df2023total.merge(
    municipio2023[['CÓDIGO DIVIPOLA', 'tasacomparendos100k']],
    on='CÓDIGO DIVIPOLA',
    how='left'
)

In [ ]:
df2023total = df2023total.drop(columns=[
    'num', 'desc1', 'desc2', 'desc3', 'desc4', 'desc5',
    'desc6', 'desc7', 'desc8', 'desc9', 'desc10',
    'concat', 'art', 'art.desc', 'tasax100000', 'POBLACION'
])

In [ ]:
df2023total.to_csv('df2023total.csv', index=False)